<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/enzo%2Fp10-8-clinical-expansion-preflight/73_P10_8_clinical_and_technical_viability_gate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 73 — P10.8: compuerta de viabilidad clínica y técnica

Este notebook consolida la evidencia de los Notebooks 69–72 y decide, de forma
conservadora, qué líneas pueden continuar como **protocolo de investigación**, cuáles
quedan **bloqueadas por falta de anotaciones o semántica**, y cuáles están **cerradas
para evitar reentrenamiento**.

Reglas:

- no entrena;
- no carga ni deserializa `.pt`;
- no abre tests sellados;
- no crea ground truth clínico;
- no convierte coincidencias textuales en etiquetas;
- no habilita diagnóstico autónomo;
- no reabre tareas ya entrenadas en P10.6/P10.7.

La salida habilita únicamente el diseño del Notebook 74 para mediciones geométricas
cuando corresponda. No habilita automatización clínica ni entrenamiento.


In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Entorno no Colab")


Mounted at /content/drive


In [3]:
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

ROOT = Path(
    os.getenv(
        "PFI_ROOT",
        "/content/drive/MyDrive/PFI_MVP",
    )
)
PREF = Path(
    os.getenv(
        "PFI_P10_8_PREFLIGHT_ROOT",
        str(
            ROOT
            / "results"
            / "P10_8_clinical_expansion_preflight"
        ),
    )
)

N72_ROOT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK72_ROOT",
        str(
            PREF
            / "alkafri_mask_semantics_and_label_normalization"
        ),
    )
)
OUT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK73_ROOT",
        str(PREF / "viability_gate"),
    )
)

REQUIRED_MARKERS = {
    "notebook69": PREF / "NOTEBOOK_69_COMPLETE.json",
    "notebook70": PREF / "NOTEBOOK_70_COMPLETE.json",
    "notebook71": Path(
        os.getenv(
            "PFI_P10_8_NOTEBOOK71_ROOT",
            str(PREF / "sudirman_alkafri_audit_v2"),
        )
    )
    / "NOTEBOOK_71_COMPLETE.json",
    "notebook72": N72_ROOT / "NOTEBOOK_72_COMPLETE.json",
}

markers: dict[str, dict[str, Any]] = {}

for name, path in REQUIRED_MARKERS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Falta marcador requerido {name}: {path}"
        )

    markers[name] = json.loads(
        path.read_text(encoding="utf-8")
    )

expected_statuses = {
    "notebook69": "NOTEBOOK_69_COMPLETE",
    "notebook70": "NOTEBOOK_70_COMPLETE",
    "notebook71": "NOTEBOOK_71_COMPLETE",
    "notebook72": "NOTEBOOK_72_COMPLETE",
}

for name, expected_status in expected_statuses.items():
    actual_status = markers[name].get("status")

    if actual_status != expected_status:
        raise RuntimeError(
            f"{name} inválido: {actual_status!r}; "
            f"se esperaba {expected_status!r}"
        )

for name, marker in markers.items():
    if marker.get("trainingExecuted") is not False:
        raise RuntimeError(
            f"{name} no declara trainingExecuted=false"
        )

    # Notebook 70 puede no incluir este campo.
    # Un valor ausente se acepta; un True explícito bloquea.
    if marker.get("weightsDeserialized", False) is not False:
        raise RuntimeError(
            f"{name} declara weightsDeserialized distinto de false"
        )

if markers["notebook71"].get("inventoryTruncated") is not False:
    raise RuntimeError(
        "El inventario válido del Notebook 71 quedó truncado"
    )

if int(
    markers["notebook71"].get("indexedFileCount", 0)
) < 1000:
    raise RuntimeError(
        "Notebook 71 no auditó la fuente primaria completa"
    )

if markers["notebook72"].get(
    "trainingAuthorized"
) is not False:
    raise RuntimeError(
        "Notebook 72 no mantiene trainingAuthorized=false"
    )

print("Marcadores 69–72 verificados.")
print("Salida Notebook 73:", OUT)


Marcadores 69–72 verificados.
Salida Notebook 73: /content/drive/MyDrive/PFI_MVP/results/P10_8_clinical_expansion_preflight/viability_gate


In [4]:
def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

INPUT_PATHS = {
    "candidateSupport":
        N72_ROOT
        / "candidate_finding_support_matrix_v1.csv",
    "semanticsRegistry":
        N72_ROOT
        / "d_token_semantics_registry_v1.csv",
    "maskInventorySummary":
        N72_ROOT
        / "mask_inventory_summary_v1.csv",
    "pairingRegistry":
        N72_ROOT
        / "manual_processed_pairing_registry_v1.csv",
    "pairComparison":
        N72_ROOT
        / "manual_processed_pixel_comparison_v1.csv",
    "xcfAudit":
        N72_ROOT
        / "xcf_header_and_layer_audit_v1.csv",
    "documentAudit":
        N72_ROOT
        / "documentation_and_code_term_audit_v1.csv",
}

for name, path in INPUT_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Falta salida requerida de Notebook 72 "
            f"({name}): {path}"
        )

input_hashes = {
    name: sha256_file(path)
    for name, path in INPUT_PATHS.items()
}

candidate_support_72 = pd.read_csv(
    INPUT_PATHS["candidateSupport"]
)
semantics_72 = pd.read_csv(
    INPUT_PATHS["semanticsRegistry"]
)
inventory_72 = pd.read_csv(
    INPUT_PATHS["maskInventorySummary"]
)
pairing_72 = pd.read_csv(
    INPUT_PATHS["pairingRegistry"]
)
comparison_72 = pd.read_csv(
    INPUT_PATHS["pairComparison"]
)
xcf_72 = pd.read_csv(
    INPUT_PATHS["xcfAudit"]
)
documents_72 = pd.read_csv(
    INPUT_PATHS["documentAudit"]
)

if (
    "trainingAuthorized"
    not in candidate_support_72.columns
):
    raise RuntimeError(
        "candidate support no contiene trainingAuthorized"
    )

truthy_training = (
    candidate_support_72["trainingAuthorized"]
    .astype(str)
    .str.lower()
    .isin({"true", "1", "yes"})
)

if truthy_training.any():
    raise RuntimeError(
        "Notebook 72 contiene una tarea con entrenamiento "
        "autorizado; la compuerta debe detenerse"
    )

required_tokens = {"D3", "D4", "D5"}
observed_tokens = set(
    semantics_72["dToken"].dropna().astype(str)
)

if not required_tokens.issubset(observed_tokens):
    raise RuntimeError(
        "Falta algún token D3/D4/D5 en el registro "
        "semántico"
    )

exact_mapping = (
    semantics_72["documentedExactMappingFound"]
    .astype(str)
    .str.lower()
    .isin({"true", "1", "yes"})
)

print("Hashes de entradas:")
display(
    pd.DataFrame(
        [
            {
                "inputName": name,
                "sha256": value,
            }
            for name, value in input_hashes.items()
        ]
    )
)

print(
    "Mapeos semánticos exactos documentados:",
    int(exact_mapping.sum()),
)


Hashes de entradas:


,inputName,sha256
0,candidateSupport,df4b8f03f591b5c999ea6942a6347dabf0026b260a08c8...
1,semanticsRegistry,4edbaf2edd9073f567f1cbc2f0152eb35c2f754fdd4187...
2,maskInventorySummary,de0d81ca13eaabe5726d7516ecd274e2059ff15b11b2e2...
3,pairingRegistry,ea1722d486abe817d0148a34af3e295ebd6535aaa3d6f2...
4,pairComparison,ad0d3ade135f27d0af898b3e9fa431e68fcba2c9e8b812...
5,xcfAudit,f1e12ffe8ebe2856bbacbd7e98ea607227b0132b2a6ab7...
6,documentAudit,20dca6b8b7b45151bcc0d2ef3c61ac00b03b6a0625819e...


Mapeos semánticos exactos documentados: 0


In [5]:
# Guardia de tareas ya entrenadas o formalmente cerradas.
TRAINED_OR_CLOSED_TASKS = {
    "central_canal_stenosis": {
        "program": "P10.6",
        "guardReason":
            "Tarea ya entrenada/evaluada; no reentrenar",
    },
    "neural_foraminal_narrowing": {
        "program": "P10.6",
        "guardReason":
            "Tarea ya entrenada/evaluada; no reentrenar",
    },
    "subarticular_stenosis": {
        "program": "P10.6",
        "guardReason":
            "Checkpoint congelado y evaluado; no reentrenar",
    },
    "pfirrmann_grade": {
        "program": "P10.7",
        "guardReason":
            "Tarea multitarea ya entrenada/evaluada",
    },
    "modic_change": {
        "program": "P10.7",
        "guardReason":
            "Tarea multitarea ya entrenada/evaluada",
    },
    "upper_endplate_change": {
        "program": "P10.7",
        "guardReason":
            "Tarea multitarea ya entrenada/evaluada",
    },
    "lower_endplate_change": {
        "program": "P10.7",
        "guardReason":
            "Tarea multitarea ya entrenada/evaluada",
    },
    "spondylolisthesis": {
        "program": "P10.7",
        "guardReason":
            "Tarea multitarea ya entrenada/evaluada; "
            "no crear otro entrenamiento",
    },
    "disc_herniation": {
        "program": "P10.7",
        "guardReason":
            "Tarea multitarea ya entrenada/evaluada; "
            "no crear otro entrenamiento",
    },
    "disc_narrowing": {
        "program": "P10.7",
        "guardReason":
            "Tarea multitarea ya entrenada/evaluada",
    },
    "disc_bulging": {
        "program": "P10.7",
        "guardReason":
            "Tarea multitarea ya entrenada/evaluada; "
            "no crear otro entrenamiento",
    },
}

retraining_guard = pd.DataFrame(
    [
        {
            "task": task,
            "program": values["program"],
            "guardReason": values["guardReason"],
            "retrainingAuthorized": False,
            "checkpointLoaded": False,
        }
        for task, values
        in sorted(TRAINED_OR_CLOSED_TASKS.items())
    ]
)

display(retraining_guard)


,task,program,guardReason,retrainingAuthorized,checkpointLoaded
0,central_canal_stenosis,P10.6,Tarea ya entrenada/evaluada; no reentrenar,False,False
1,disc_bulging,P10.7,Tarea multitarea ya entrenada/evaluada; no cre...,False,False
2,disc_herniation,P10.7,Tarea multitarea ya entrenada/evaluada; no cre...,False,False
3,disc_narrowing,P10.7,Tarea multitarea ya entrenada/evaluada,False,False
4,lower_endplate_change,P10.7,Tarea multitarea ya entrenada/evaluada,False,False
5,modic_change,P10.7,Tarea multitarea ya entrenada/evaluada,False,False
6,neural_foraminal_narrowing,P10.6,Tarea ya entrenada/evaluada; no reentrenar,False,False
7,pfirrmann_grade,P10.7,Tarea multitarea ya entrenada/evaluada,False,False
8,spondylolisthesis,P10.7,Tarea multitarea ya entrenada/evaluada; no cre...,False,False
9,subarticular_stenosis,P10.6,Checkpoint congelado y evaluado; no reentrenar,False,False


In [6]:
CANDIDATES = [
    {
        "findingType": "facet_hypertrophy",
        "candidateKind": "new_clinical_finding",
        "requiredPlane": "axial_or_multiplanar",
        "targetNextStep": "none",
    },
    {
        "findingType":
            "ligamentum_flavum_hypertrophy",
        "candidateKind": "new_clinical_finding",
        "requiredPlane": "axial_or_multiplanar",
        "targetNextStep": "none",
    },
    {
        "findingType": "annular_tear",
        "candidateKind": "new_clinical_finding",
        "requiredPlane": "sagittal_and_axial",
        "targetNextStep": "none",
    },
    {
        "findingType": "nerve_root_compression",
        "candidateKind": "new_clinical_finding",
        "requiredPlane": "sagittal_and_axial",
        "targetNextStep": "none",
    },
    {
        "findingType": "epidural_fat",
        "candidateKind": "anatomical_structure",
        "requiredPlane": "axial_or_multiplanar",
        "targetNextStep": "none",
    },
    {
        "findingType": "disc_height",
        "candidateKind": "geometry_measurement",
        "requiredPlane": "sagittal",
        "targetNextStep":
            "NOTEBOOK_74_PROTOCOL_ONLY",
    },
    {
        "findingType": "ap_diameter",
        "candidateKind": "geometry_measurement",
        "requiredPlane": "axial",
        "targetNextStep":
            "NOTEBOOK_74_PROTOCOL_ONLY",
    },
    {
        "findingType": "spondylolisthesis",
        "candidateKind": "already_trained_task",
        "requiredPlane": "sagittal",
        "targetNextStep": "none",
    },
    {
        "findingType": "disc_herniation",
        "candidateKind": "already_trained_task",
        "requiredPlane": "sagittal_and_axial",
        "targetNextStep":
            "NOTEBOOK_75_REVIEW_ONLY_NO_RETRAINING",
    },
    {
        "findingType": "disc_bulging",
        "candidateKind": "already_trained_task",
        "requiredPlane": "sagittal_and_axial",
        "targetNextStep": "none",
    },
]

support_by_finding: dict[str, dict[str, Any]] = {}

for _, row in candidate_support_72.iterrows():
    support_by_finding[str(row["findingType"])] = (
        row.to_dict()
    )

exact_semantic_mapping_count = int(
    exact_mapping.sum()
)
semantic_mapping_validated = (
    exact_semantic_mapping_count == 3
)

gate_rows = []

for candidate in CANDIDATES:
    finding = candidate["findingType"]
    prior_support = support_by_finding.get(
        finding,
        {},
    )
    already_trained = (
        finding in TRAINED_OR_CLOSED_TASKS
    )

    documentation_term_present = str(
        prior_support.get(
            "documentationTermPresent",
            False,
        )
    ).lower() in {"true", "1", "yes"}

    mask_mapping_validated = str(
        prior_support.get(
            "maskClassMappingValidated",
            False,
        )
    ).lower() in {"true", "1", "yes"}

    alignment_validated = str(
        prior_support.get(
            "caseSeriesLevelSliceAlignmentValidated",
            False,
        )
    ).lower() in {"true", "1", "yes"}

    if already_trained:
        decision = (
            "CLOSED_NO_RETRAINING_ALREADY_TRAINED"
        )
        rationale = (
            TRAINED_OR_CLOSED_TASKS[finding][
                "guardReason"
            ]
        )
        protocol_design_allowed = (
            candidate["targetNextStep"]
            == "NOTEBOOK_75_REVIEW_ONLY_NO_RETRAINING"
        )
    elif (
        candidate["candidateKind"]
        == "geometry_measurement"
    ):
        decision = (
            "PROTOCOL_ONLY_BLOCKED_PENDING_"
            "ANATOMICAL_SEMANTICS_AND_ALIGNMENT"
        )
        rationale = (
            "Puede definirse el protocolo matemático y "
            "de revisión, pero no automatizarse ni "
            "validarse clínicamente con la evidencia "
            "actual."
        )
        protocol_design_allowed = True
    else:
        decision = (
            "BLOCKED_CURRENT_DATASET_ANNOTATION_"
            "NOT_DEMONSTRATED"
        )
        rationale = (
            "No existe asociación demostrada entre "
            "máscara, caso, serie, nivel/corte y "
            "significado clínico del hallazgo."
        )
        protocol_design_allowed = False

    viable_for_training = bool(
        not already_trained
        and semantic_mapping_validated
        and documentation_term_present
        and mask_mapping_validated
        and alignment_validated
    )

    # La compuerta P10.8 nunca autoriza entrenamiento
    # automáticamente, aunque todos los requisitos
    # llegaran a ser verdaderos.
    training_authorized = False

    gate_rows.append(
        {
            "findingType": finding,
            "candidateKind":
                candidate["candidateKind"],
            "requiredPlane":
                candidate["requiredPlane"],
            "documentationTermPresent":
                documentation_term_present,
            "exactMaskSemanticsValidated":
                semantic_mapping_validated,
            "maskClassMappingValidated":
                mask_mapping_validated,
            "caseSeriesLevelSliceAlignmentValidated":
                alignment_validated,
            "alreadyTrainedOrClosed":
                already_trained,
            "viableForTrainingByEvidence":
                viable_for_training,
            "trainingAuthorized":
                training_authorized,
            "protocolDesignAllowed":
                protocol_design_allowed,
            "targetNextStep":
                candidate["targetNextStep"],
            "gateDecision": decision,
            "rationale": rationale,
        }
    )

viability_gate = pd.DataFrame(gate_rows)

display(viability_gate)


,findingType,candidateKind,requiredPlane,documentationTermPresent,exactMaskSemanticsValidated,maskClassMappingValidated,caseSeriesLevelSliceAlignmentValidated,alreadyTrainedOrClosed,viableForTrainingByEvidence,trainingAuthorized,protocolDesignAllowed,targetNextStep,gateDecision,rationale
0,facet_hypertrophy,new_clinical_finding,axial_or_multiplanar,False,False,False,False,False,False,False,False,none,BLOCKED_CURRENT_DATASET_ANNOTATION_NOT_DEMONST...,"No existe asociación demostrada entre máscara,..."
1,ligamentum_flavum_hypertrophy,new_clinical_finding,axial_or_multiplanar,False,False,False,False,False,False,False,False,none,BLOCKED_CURRENT_DATASET_ANNOTATION_NOT_DEMONST...,"No existe asociación demostrada entre máscara,..."
2,annular_tear,new_clinical_finding,sagittal_and_axial,False,False,False,False,False,False,False,False,none,BLOCKED_CURRENT_DATASET_ANNOTATION_NOT_DEMONST...,"No existe asociación demostrada entre máscara,..."
3,nerve_root_compression,new_clinical_finding,sagittal_and_axial,False,False,False,False,False,False,False,False,none,BLOCKED_CURRENT_DATASET_ANNOTATION_NOT_DEMONST...,"No existe asociación demostrada entre máscara,..."
4,epidural_fat,anatomical_structure,axial_or_multiplanar,False,False,False,False,False,False,False,False,none,BLOCKED_CURRENT_DATASET_ANNOTATION_NOT_DEMONST...,"No existe asociación demostrada entre máscara,..."
5,disc_height,geometry_measurement,sagittal,False,False,False,False,False,False,False,True,NOTEBOOK_74_PROTOCOL_ONLY,PROTOCOL_ONLY_BLOCKED_PENDING_ANATOMICAL_SEMAN...,Puede definirse el protocolo matemático y de r...
6,ap_diameter,geometry_measurement,axial,False,False,False,False,False,False,False,True,NOTEBOOK_74_PROTOCOL_ONLY,PROTOCOL_ONLY_BLOCKED_PENDING_ANATOMICAL_SEMAN...,Puede definirse el protocolo matemático y de r...
7,spondylolisthesis,already_trained_task,sagittal,False,False,False,False,True,False,False,False,none,CLOSED_NO_RETRAINING_ALREADY_TRAINED,Tarea multitarea ya entrenada/evaluada; no cre...
8,disc_herniation,already_trained_task,sagittal_and_axial,False,False,False,False,True,False,False,True,NOTEBOOK_75_REVIEW_ONLY_NO_RETRAINING,CLOSED_NO_RETRAINING_ALREADY_TRAINED,Tarea multitarea ya entrenada/evaluada; no cre...
9,disc_bulging,already_trained_task,sagittal_and_axial,False,False,False,False,True,False,False,False,none,CLOSED_NO_RETRAINING_ALREADY_TRAINED,Tarea multitarea ya entrenada/evaluada; no cre...


In [7]:
measurement_protocol = viability_gate[
    viability_gate["candidateKind"]
    == "geometry_measurement"
][
    [
        "findingType",
        "requiredPlane",
        "protocolDesignAllowed",
        "trainingAuthorized",
        "targetNextStep",
        "gateDecision",
        "rationale",
    ]
].copy()

measurement_protocol["automaticMeasurementValidated"] = False
measurement_protocol["clinicalThresholdFrozen"] = False
measurement_protocol["professionalReviewRequired"] = True
measurement_protocol["notClinicalDiagnosis"] = True

blocked_findings = viability_gate[
    viability_gate["gateDecision"].str.startswith(
        "BLOCKED_"
    )
].copy()

closed_no_retraining = viability_gate[
    viability_gate["gateDecision"].str.startswith(
        "CLOSED_NO_RETRAINING"
    )
].copy()

decision_counts = (
    viability_gate["gateDecision"]
    .value_counts()
    .rename_axis("gateDecision")
    .reset_index(name="candidateCount")
)

print("Conteo de decisiones:")
display(decision_counts)

print("Candidatos de protocolo geométrico:")
display(measurement_protocol)


Conteo de decisiones:


,gateDecision,candidateCount
0,BLOCKED_CURRENT_DATASET_ANNOTATION_NOT_DEMONST...,5
1,CLOSED_NO_RETRAINING_ALREADY_TRAINED,3
2,PROTOCOL_ONLY_BLOCKED_PENDING_ANATOMICAL_SEMAN...,2


Candidatos de protocolo geométrico:


,findingType,requiredPlane,protocolDesignAllowed,trainingAuthorized,targetNextStep,gateDecision,rationale,automaticMeasurementValidated,clinicalThresholdFrozen,professionalReviewRequired,notClinicalDiagnosis
5,disc_height,sagittal,True,False,NOTEBOOK_74_PROTOCOL_ONLY,PROTOCOL_ONLY_BLOCKED_PENDING_ANATOMICAL_SEMAN...,Puede definirse el protocolo matemático y de r...,False,False,True,True
6,ap_diameter,axial,True,False,NOTEBOOK_74_PROTOCOL_ONLY,PROTOCOL_ONLY_BLOCKED_PENDING_ANATOMICAL_SEMAN...,Puede definirse el protocolo matemático y de r...,False,False,True,True


In [8]:
OUT.mkdir(parents=True, exist_ok=True)

paths = {
    "viabilityGate":
        OUT / "p10_8_viability_gate_v1.csv",
    "measurementProtocolCandidates":
        OUT
        / "measurement_protocol_candidates_v1.csv",
    "retrainingGuard":
        OUT / "retraining_guard_decisions_v1.csv",
    "blockedFindings":
        OUT / "blocked_findings_v1.csv",
    "closedNoRetraining":
        OUT / "closed_no_retraining_v1.csv",
    "inputHashes":
        OUT / "notebook72_input_hashes_v1.csv",
    "summary":
        OUT / "NOTEBOOK_73_SUMMARY.json",
}

viability_gate.to_csv(
    paths["viabilityGate"],
    index=False,
)
measurement_protocol.to_csv(
    paths["measurementProtocolCandidates"],
    index=False,
)
retraining_guard.to_csv(
    paths["retrainingGuard"],
    index=False,
)
blocked_findings.to_csv(
    paths["blockedFindings"],
    index=False,
)
closed_no_retraining.to_csv(
    paths["closedNoRetraining"],
    index=False,
)
pd.DataFrame(
    [
        {
            "inputName": name,
            "sha256": value,
        }
        for name, value in input_hashes.items()
    ]
).to_csv(
    paths["inputHashes"],
    index=False,
)

pt_files_in_output = list(OUT.rglob("*.pt"))

if pt_files_in_output:
    raise RuntimeError(
        "La carpeta de salida del Notebook 73 contiene "
        "archivos .pt inesperados"
    )

summary = {
    "schemaVersion":
        "pfi.p10-8.notebook-73-summary.v1",
    "generatedAtUtc":
        datetime.now(timezone.utc).isoformat(),
    "trainingExecuted": False,
    "weightsDeserialized": False,
    "internalTestAccessed": False,
    "officialHiddenTestAccessed": False,
    "patientIdentifiersExported": False,
    "clinicalGroundTruthCreated": False,
    "clinicalThresholdsFrozen": False,
    "automaticMeasurementValidated": False,
    "trainingAuthorized": False,
    "candidateCount":
        int(len(viability_gate)),
    "trainingAuthorizedCount":
        int(
            viability_gate[
                "trainingAuthorized"
            ].sum()
        ),
    "viableForTrainingByEvidenceCount":
        int(
            viability_gate[
                "viableForTrainingByEvidence"
            ].sum()
        ),
    "blockedCurrentDatasetCount":
        int(len(blocked_findings)),
    "closedNoRetrainingCount":
        int(len(closed_no_retraining)),
    "protocolDesignCandidateCount":
        int(
            viability_gate[
                "protocolDesignAllowed"
            ].sum()
        ),
    "geometryProtocolCandidateCount":
        int(len(measurement_protocol)),
    "exactDocumentedTokenMappings":
        exact_semantic_mapping_count,
    "semanticMappingValidated":
        semantic_mapping_validated,
    "retrainingGuardTaskCount":
        int(len(retraining_guard)),
    "outputPtFileCount":
        len(pt_files_in_output),
    "nextRequiredGate":
        "NOTEBOOK_74_GEOMETRY_MEASUREMENT_PROTOCOL",
    "notClinicalDiagnosis": True,
}

write_json(paths["summary"], summary)

marker = {
    "schemaVersion":
        "pfi.p10-8.notebook-73-complete.v1",
    "status": "NOTEBOOK_73_COMPLETE",
    **summary,
    "outputs": {
        key: str(value)
        for key, value in paths.items()
    },
}

write_json(
    OUT / "NOTEBOOK_73_COMPLETE.json",
    marker,
)

print(json.dumps(marker, indent=2, ensure_ascii=False))
print("NOTEBOOK_73_COMPLETE")


{
  "schemaVersion": "pfi.p10-8.notebook-73-summary.v1",
  "status": "NOTEBOOK_73_COMPLETE",
  "generatedAtUtc": "2026-08-07T02:35:54.576670+00:00",
  "trainingExecuted": false,
  "weightsDeserialized": false,
  "internalTestAccessed": false,
  "officialHiddenTestAccessed": false,
  "patientIdentifiersExported": false,
  "clinicalGroundTruthCreated": false,
  "clinicalThresholdsFrozen": false,
  "automaticMeasurementValidated": false,
  "trainingAuthorized": false,
  "candidateCount": 10,
  "trainingAuthorizedCount": 0,
  "viableForTrainingByEvidenceCount": 0,
  "blockedCurrentDatasetCount": 5,
  "closedNoRetrainingCount": 3,
  "protocolDesignCandidateCount": 3,
  "geometryProtocolCandidateCount": 2,
  "exactDocumentedTokenMappings": 0,
  "semanticMappingValidated": false,
  "retrainingGuardTaskCount": 11,
  "outputPtFileCount": 0,
  "nextRequiredGate": "NOTEBOOK_74_GEOMETRY_MEASUREMENT_PROTOCOL",
  "notClinicalDiagnosis": true,
  "outputs": {
    "viabilityGate": "/content/drive/MyDri

## Interpretación obligatoria

- `PROTOCOL_ONLY...` permite diseñar fórmulas, entradas, salidas y revisión profesional;
  no permite afirmar que la medición automática esté validada.
- `BLOCKED_CURRENT_DATASET...` indica que la anotación requerida no fue demostrada.
- `CLOSED_NO_RETRAINING...` protege tareas ya entrenadas en P10.6/P10.7.
- El Notebook 74 debe limitarse a protocolos geométricos para altura discal y diámetro
  anteroposterior, con revisión profesional y sin umbrales clínicos congelados.
- Ningún resultado debe presentarse como diagnóstico clínico autónomo.
